# Laboratory 10: Quantum Optics on the Bench — Analyzing Your Data

At the bench you counted clicks with the Thorlabs EDU-QOP1 Quantum Optics Kit: single-photon
detectors, a time tagger that finds coincidences inside a window $\Delta t$, and the kit
software that prints $g^{(2)}$. This notebook recomputes those numbers from the counts, using
only numpy, pandas and matplotlib (one optional cell at the very end uses Qiskit). The
questions driving the four parts are the bench guide's:

1. **Part A** — is a very dim laser a single-photon source? (the manual's Table 1)
2. **Part B** — does the crystal really make pairs? (the manual's Table 2)
3. **Part C** — can one photon take both exits, and do chance coincidences explain what is
   left? (Table 2 again)
4. **Part D** — does a single photon interfere with itself? (a Michelson scan turned into a
   wavelength)

After Part C and after Part D a **Your data** cell runs the same computation on your
worksheet (W1–W4). Two optional sections close the notebook: what entanglement would add
(numpy only), and the pair states as Qiskit circuits.

> **SYNTHETIC data.** The manual shows its Michelson scans only as figures, and figures are
> never read off here. Until real scans from the bench exist, Part D runs on a **SYNTHETIC**
> scan drawn from a model with seed 1010 (`lab10_michelson_SYNTHETIC_seed1010.csv`). It is
> **not** a real bench run: replace it with your scan (Part D shows how). Parts A–C use the
> manual's printed example measurements.

**What you need.** The lecture series *Fundamental Concepts of Quantum Technologies*, and
nothing else: every term is defined where it is first used.

**Conventions.** Polarization follows the course convention: $|H\rangle$ is horizontal and
$|V\rangle$ vertical. Numbers of photons are written in words ("one photon", "vacuum"), never
as kets. Everything in Parts A–D is computed from click counts.

**Source.** Thorlabs, *EDU-QOP1(/M) Quantum Optics Kit User Guide*, Rev. D (2025-10-25),
cited as "manual §x". Its §9.1 Table 1 and §9.4 Table 2 are transcribed with attribution in
`lab10_manual_tables.csv` (printed below); everything derived from them is recomputed here.

In [ ]:
"""
Setup: libraries, the data loader and the formulas every part uses.
Run this cell first. No quantum library is needed until the optional last cell.
"""
import io
import math
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 1010     # the seed of the SYNTHETIC scan; the optional Qiskit cell reuses it
RAW_BASE = "https://raw.githubusercontent.com/Quantum-Plus-Labs/quantum-labs-2026/main/lab10/"
LOADED_FROM = {}    # file name -> where it was read from

DATA_COLOR, PREDICTION_COLOR = "#2a78d6", "#eb6834"    # charts: blue = data, orange = prediction


def load_csv(name):
    """Read a lab CSV: the file next to this notebook if present, else the course repository.

    Lines starting with '#' carry the source and provenance: they are returned as notes and
    skipped by pandas (comment='#'). Every value is read as text, so a blank cell stays "".
    """
    local = Path(name)
    if local.is_file():
        text, source = local.read_text(encoding="utf-8"), str(local)
    else:
        with urllib.request.urlopen(RAW_BASE + name) as response:
            text, source = response.read().decode("utf-8"), RAW_BASE + name
    LOADED_FROM[name] = source
    notes = [line.rstrip().removeprefix("# ").removeprefix("#")
             for line in text.splitlines() if line.startswith("#")]
    table = pd.read_csv(io.StringIO(text), comment="#", dtype=str, keep_default_na=False)
    return table, notes


def to_number(column):
    """Text to float: a blank or unreadable cell becomes nan (never an error)."""
    return pd.to_numeric(column.astype(str).str.strip(), errors="coerce").astype(float)


def ratio(top, bottom):
    """top / bottom as a float; nan or inf instead of an error when an input is blank or 0."""
    with np.errstate(divide="ignore", invalid="ignore"):
        return float(np.float64(top) / np.float64(bottom))


def g2_hbt(r_a, r_b, r_ab, window_s):
    """g2_HBT = R_AB / (R_A R_B Δt): coincidences over accidentals (manual eq. 98)."""
    return ratio(r_ab, r_a * r_b * window_s)


def g2_ps(r_t, r_a, r_ta, window_s):
    """g2_PS = R_TA / (R_T R_A Δt): T&A coincidences over their accidentals (manual §9.2)."""
    return ratio(r_ta, r_t * r_a * window_s)


def g2_gra(n_t, n_ta, n_tb, n_tab):
    """g2_GRA = N_TAB N_T / (N_TA N_TB) (manual eq. 101): counts, so the measurement time cancels."""
    return ratio(n_tab * n_t, n_ta * n_tb)


def gra_sigma(g2, n_tab):
    """sigma ≈ g2_GRA / sqrt(N_TAB): a count of N clicks scatters by about sqrt(N) (triples only)."""
    return ratio(g2, np.sqrt(np.float64(n_tab)))


def accidental_triples(r_t, r_a, r_b, r_ta, r_tb, window_s):
    """Chance triples per second (manual eqs. 105-106): (R3, R21).

    R3  = R_T R_A R_B Δt^2           three unrelated clicks in one window
    R21 = (R_TA R_B + R_TB R_A) Δt   a true pair plus one unrelated click
    """
    return r_t * r_a * r_b * window_s ** 2, (r_ta * r_b + r_tb * r_a) * window_s


def wavelength_from_maxima(x_first_nm, x_last_nm, n_max):
    """lambda = 2 (x_last - x_first) / (N_max - 1): twice the mean spacing of the fringe maxima."""
    return ratio(2.0 * (x_last_nm - x_first_nm), n_max - 1)


def verdict(value, test):
    """'yes' or 'no' from test(value), or 'not filled in' when the value is nan."""
    if value is None or np.isnan(value):
        return "not filled in"
    return "yes" if test(value) else "no"


print("Setup complete. Seed of the SYNTHETIC scan:", SEED)

---
## A · Is a Very Dim Laser a Single-Photon Source?

**The question first:** *the manual dims the alignment laser until, on average, far less than
one photon is inside the setup at any moment. Is that a single-photon source?*

**Why this step:** a **beam splitter** is a half-transparent mirror with two outputs; detectors
A and B sit behind them. A **click** is one pulse of a single-photon detector, and a
**coincidence** is two (or three) detectors clicking in the same **window** $\Delta t$ (5 ns,
set in the software). With $P(X)$ the probability that detector $X$ clicks in one window,
$P(X) = R_X\,\Delta t$ for a click rate $R_X$ (clicks per second), and

$$g^{(2)} = \frac{P(A \text{ and } B)}{P(A)\,P(B)}
  \quad\Longrightarrow\quad
  g^{(2)}_{\mathrm{HBT}} = \frac{R_{AB}}{R_A R_B\,\Delta t}$$

(manual eq. 98; HBT = Hanbury Brown and Twiss). The **accidentals** $R_A R_B\,\Delta t$ are the
coincidences expected by pure chance if A and B click independently, so $g^{(2)}$ =
coincidences / accidentals: 1 = no correlation, below 1 = fewer coincidences than chance,
above 1 = more.

**Poisson light** is light whose photons arrive independently at random times, like
raindrops; a laser, however dim, is Poisson light. Each of its photons picks its exit on its
own, so A and B click independently and $g^{(2)}_{\mathrm{HBT}} = 1$ exactly, at every
brightness. A single photon reaches A *or* B, never both: $g^{(2)} = 0$.

The manual also estimates how many photons are inside the setup at any moment,
$n = l f / c$, from the length $l$ of the setup, the number $f$ of photons per second and the
speed of light $c$.

In [ ]:
"""
Part A - Table 1 recomputed: the accidentals and g2_HBT; the manual's photon estimate.
"""
manual, notes = load_csv("lab10_manual_tables.csv")
print("\n".join(notes), "\n")
manual["number"] = to_number(manual["value"])
assert manual["number"].notna().all()


def manual_values(source):
    """{quantity: value in SI units} of one source in the manual file."""
    rows = manual[manual["source"] == source]
    return dict(zip(rows["quantity"], rows["number"]))


T1, SHOCK, T2 = manual_values("Table 1"), manual_values("§9.1 text"), manual_values("Table 2")
PRINTED = dict(zip(manual["source"] + " " + manual["quantity"], manual["printed"]))

accidentals_t1 = T1["R_A"] * T1["R_B"] * T1["window"]
g2_hbt_t1 = g2_hbt(T1["R_A"], T1["R_B"], T1["R_AB"], T1["window"])
print(f"Table 1, window {PRINTED['Table 1 window']}:")
print(f"  accidentals R_A R_B Δt = {accidentals_t1:.4f} Hz;  measured R_AB = {PRINTED['Table 1 R_AB']}")
print(f"  g2_HBT = R_AB / accidentals = {g2_hbt_t1:.7f}, i.e. {g2_hbt_t1:.4f}  "
      f"(the manual prints {PRINTED['Table 1 g2_HBT']})")
assert abs(g2_hbt_t1 - T1["g2_HBT"]) < 1e-4       # the printed value, within its rounding

photons_in_setup = SHOCK["l"] * SHOCK["f"] / SHOCK["c"]
print(f"\nphotons inside the setup, n = l f / c = {PRINTED['§9.1 text l']} x {PRINTED['§9.1 text f']}"
      f" / {PRINTED['§9.1 text c']} = {photons_in_setup:.3f}")
print(f"far less than one photon, yet g2_HBT = {g2_hbt_t1:.4f}, not 0: rare photons are not single "
      "photons, and a dim laser is still Poisson light.")

---
## B · Does the Crystal Really Make Pairs?

**The question first:** *with the pump laser on, detectors T and A both see light from the
crystal. Do they click together more often than chance?*

**Why this step:** a **pair source** is a crystal that, pumped at 405 nm, emits two photons
(about 810 nm) at once: one toward detector T, one toward the beam splitter. If it really makes
pairs, T and A click together far more often than their accidentals $R_T R_A\,\Delta t$:

$$g^{(2)}_{\mathrm{PS}} = \frac{R_{TA}}{R_T R_A\,\Delta t} .$$

The manual expects at least 30 (§9.2). Table 2 prints counts over its measurement time, so the
rates are $R_X = N_X / \text{time}$. It prints no window: we assume the standard
$\Delta t = 5$ ns (manual §7.4.5), the window of Table 1. Of the numbers Table 2 prints only
$g^{(2)}_{\mathrm{PS}}$ depends on this assumption, and the recomputed value corroborates it;
the accidentals of Part C use the same window.

In [ ]:
"""
Part B - Table 2 recomputed: the rates and the pair-source g2_PS at the standard window.
"""
STANDARD_WINDOW_S = 5e-9    # manual §7.4.5; Table 2 prints no window (an assumption)
assert STANDARD_WINDOW_S == T1["window"]                 # the window of Table 1
G2_PS_MIN = 30              # manual §9.2: expect at least 30

rates_t2 = {x: T2[f"N_{x}"] / T2["time"] for x in ("T", "A", "B", "TA", "TB", "TAB")}
print(f"Table 2, rates R_X = N_X / time (time {PRINTED['Table 2 time']}):")
for x, rate in rates_t2.items():
    print(f"  R_{x:<4s} = {PRINTED['Table 2 N_' + x]:>9s} / {T2['time']:g} s = {rate:>10.2f} Hz")

accidentals_ps_t2 = rates_t2["T"] * rates_t2["A"] * STANDARD_WINDOW_S
g2_ps_t2 = g2_ps(rates_t2["T"], rates_t2["A"], rates_t2["TA"], STANDARD_WINDOW_S)
print(f"\naccidentals R_T R_A Δt = {accidentals_ps_t2:.2f} Hz against R_TA = {rates_t2['TA']:.2f} Hz")
print(f"g2_PS = {g2_ps_t2:.3f}, about {g2_ps_t2:.0f}  (the manual prints {PRINTED['Table 2 g2_PS']}; "
      f"it expects at least {G2_PS_MIN})")
print(f"T&A coincidences are about {g2_ps_t2:.0f} times the accidentals: the crystal makes pairs.")
assert abs(g2_ps_t2 - T2["g2_PS"]) < 0.6 and g2_ps_t2 >= G2_PS_MIN

---
## C · Can One Photon Take Both Exits?

**The question first:** *a click at T announces a photon on its way to the beam splitter. Can
that one photon make A and B click together?*

**Why this step:** **heralding** means that a click at T announces that its partner photon is
in the A/B arm, and A and B are counted only together with T. The software counts a triple as
a *coincidence of coincidences*: T&A and T&B both coincident (manual §4.2.2, §12.2.1). Then

$$g^{(2)}_{\mathrm{GRA}} = \frac{N_{TAB}\,N_T}{N_{TA}\,N_{TB}}$$

(manual eq. 101; GRA = Grangier, Roger and Aspect). These are counts, so the measurement time
cancels. Any classical light obeys $g^{(2)} \ge 1$ (manual eq. 75); one photon, which cannot
split, gives 0. The **standard deviation** $\sigma$ is the typical scatter of repeated results;
a count of $N$ clicks scatters by about $\sqrt N$. Taking the triples as the only noise (the
other counts are hundreds of thousands and more),
$\sigma \approx g^{(2)}_{\mathrm{GRA}}/\sqrt{N_{TAB}}$, and
$(1 - g^{(2)}_{\mathrm{GRA}})/\sigma$ says how many standard deviations the result lies below
the classical bound.

**The bench is not the model: the window $\Delta t$** (declared deviation DEV-10.2). Detectors
and time tagger resolve time only to about a nanosecond, so clicks that are unrelated but fall
in the same window fake triples. Two chance types (manual eqs. 105–106): three unrelated
clicks, $R^{(3)} = R_T R_A R_B\,\Delta t^2$, and a true pair plus one unrelated click,
$R^{(2+1)} = (R_{TA} R_B + R_{TB} R_A)\,\Delta t$. Expressed as a $g^{(2)}$,

$$g^{(2)}_{\mathrm{acc}} = \bigl(R^{(3)} + R^{(2+1)}\bigr)\,\frac{R_T}{R_{TA}\,R_{TB}} ,$$

and $\bigl(R^{(3)} + R^{(2+1)}\bigr)\times\text{time}$ is the number of chance triples. The
manual's eq. 109 keeps only $R^{(2+1)}$, the dominant type; the cell shows both. Our model is
idealized (declared deviation DEV-10.3): detector efficiency multiplies the top and the bottom
of $g^{(2)}_{\mathrm{HBT}}$ and $g^{(2)}_{\mathrm{PS}}$ alike and cancels, and in $g^{(2)}_{\mathrm{GRA}}$
so does the efficiency of A and B; a herald detector T that misses photons, however, raises the
small contribution of two pairs in one window by up to a factor 2 (still far below 1). Dark
counts and stray light enter only as accidentals.

In [ ]:
"""
Part C - Table 2 recomputed: g2_GRA, its distance from the classical bound, and the accidentals.
"""
g2_gra_t2 = g2_gra(T2["N_T"], T2["N_TA"], T2["N_TB"], T2["N_TAB"])
sigma_t2 = gra_sigma(g2_gra_t2, T2["N_TAB"])
n_sigma_t2 = (1 - g2_gra_t2) / sigma_t2
print(f"g2_GRA = N_TAB N_T / (N_TA N_TB) = {g2_gra_t2:.6f}, i.e. {g2_gra_t2:.4f}  "
      f"(the manual prints {PRINTED['Table 2 g2_GRA']})")
print(f"sigma ≈ g2_GRA / sqrt(N_TAB) = {sigma_t2:.3e}:  (1 - g2_GRA) / sigma = {n_sigma_t2:.0f} "
      "standard deviations below the classical bound 1")
assert abs(g2_gra_t2 - T2["g2_GRA"]) < 6e-5 and g2_gra_t2 < 1

r3_t2, r21_t2 = accidental_triples(rates_t2["T"], rates_t2["A"], rates_t2["B"],
                                   rates_t2["TA"], rates_t2["TB"], STANDARD_WINDOW_S)
herald_t2 = rates_t2["T"] / (rates_t2["TA"] * rates_t2["TB"])
g2_acc_t2, g2_acc_eq109_t2 = (r3_t2 + r21_t2) * herald_t2, r21_t2 * herald_t2
chance_t2, chance_eq109_t2 = (r3_t2 + r21_t2) * T2["time"], r21_t2 * T2["time"]
distance_t2 = (chance_t2 - T2["N_TAB"]) / math.sqrt(T2["N_TAB"])

print(f"\nchance triples at Δt = {STANDARD_WINDOW_S * 1e9:g} ns (rates of Table 2):")
print(pd.DataFrame(
    {"R3 (Hz)": [r3_t2, 0.0], "R21 (Hz)": [r21_t2, r21_t2], "g2_acc": [g2_acc_t2, g2_acc_eq109_t2],
     f"chance triples in {T2['time']:g} s": [chance_t2, chance_eq109_t2]},
    index=["both types, R3 + R21", "eq. 109, R21 only"]).to_string(float_format=lambda v: f"{v:.4f}"))
print(f"\nmeasured: N_TAB = {T2['N_TAB']:.0f} triples, g2_GRA = {g2_gra_t2:.4f};  "
      f"chance: {chance_t2:.0f} triples, g2_acc = {g2_acc_t2:.4f}")
print(f"(chance - measured) / sqrt(N_TAB) = {distance_t2:.2f} standard deviations")
if abs(distance_t2) <= 2:
    print("The residual g2_GRA is consistent with chance coincidences alone: no sign of the photon splitting.")
else:
    print("The triples differ from the chance estimate by more than 2 standard deviations.")

fig, ax = plt.subplots(figsize=(7.5, 3.2))
labels = ["measured N_TAB", "chance: R3 + R21", "chance: R21 only (eq. 109)"]
values = [T2["N_TAB"], chance_t2, chance_eq109_t2]
ax.barh(labels, values, height=0.55, color=[DATA_COLOR, PREDICTION_COLOR, PREDICTION_COLOR], alpha=0.9)
ax.errorbar(T2["N_TAB"], 0, xerr=math.sqrt(T2["N_TAB"]), fmt="none", ecolor="black", capsize=4)
for y, v in enumerate(values):
    ax.annotate(f"{v:.0f}", (v, y), xytext=(28, 0), textcoords="offset points", va="center")
ax.invert_yaxis()
ax.set_xlim(0, 1.2 * max(values))
ax.set_xlabel(f"triples in {T2['time']:g} s")
ax.set_title("Table 2 (manual §9.4): measured triples ± sqrt(N) against chance triples", fontsize=10)
ax.grid(axis="x", color="0.9")
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

---
## Your Data · Worksheets W1–W3

**The question first:** *do your own runs give the answers of the manual's examples?*

The cell below runs Parts A–C on your worksheet, side by side with Tables 1 and 2. Copy
`lab10_worksheet_template.csv` (in the course repository, next to this notebook) and type your
numbers into its `value` column: one number per row, in the unit of the `unit` column, with a
point for decimals and no thousands separator; leave a row blank if you did not measure it.
Save it under a new name, put it next to this notebook (in Colab: the Files panel, then
Upload), set `WORKSHEET_FILE` to its name, and rerun this cell and the W4 cell at the end of
Part D.

Left as it is, the cell reads the **empty template**: every result is then `nan` and every
comparison says "not filled in". W1 takes up to three runs at the kit setting (plus the
optional extra ND filter): the mean of the runs and its standard error, the standard deviation
of the runs divided by $\sqrt{\text{number of runs}}$, tell whether they are consistent with 1.

In [ ]:
"""
Your data, W1-W3: Parts A-C on your worksheet (the empty template by default).
"""
WORKSHEET_FILE = "lab10_worksheet_template.csv"    # <- your filled copy (same columns)

worksheet, notes = load_csv(WORKSHEET_FILE)
print("\n".join(notes), "\n")
worksheet["number"] = to_number(worksheet["value"])
typed = worksheet["value"].str.strip() != ""
unreadable = worksheet[typed & worksheet["number"].isna()]
if len(unreadable):
    print("UNREADABLE values, left out (use a point for decimals, no thousands separator):")
    print(unreadable[["sheet", "run", "quantity", "value"]].to_string(index=False), "\n")
n_filled = int(worksheet["number"].notna().sum())
print(f"{WORKSHEET_FILE}: {n_filled} of {len(worksheet)} values filled in"
      + ("" if n_filled else " - the empty template: every result below is nan (not filled in)"))

TO_SI = {"": 1.0, "counts": 1.0, "Hz": 1.0, "s": 1.0, "ns": 1e-9, "um": 1e-6, "nm": 1e-9}
bad_units = sorted(set(worksheet["unit"]) - set(TO_SI))
assert not bad_units, f"unknown unit(s) {bad_units}; allowed: {sorted(TO_SI)}"


def entry(sheet, run, quantity):
    """One worksheet value in SI units (Hz, s, m, counts); nan if blank or missing."""
    rows = worksheet[(worksheet["sheet"] == sheet) & (worksheet["run"] == str(run))
                     & (worksheet["quantity"] == quantity)]
    if len(rows) != 1:
        return math.nan
    return float(rows["number"].iloc[0]) * TO_SI[rows["unit"].iloc[0]]


window_s = entry("W1", "all", "window")
print(f"window Δt = {window_s * 1e9:g} ns (W1, used for W1-W3)")

# W1 - Part A: g2_HBT per run
w1_rows = [{"run": "Table 1", "R_A": T1["R_A"], "R_B": T1["R_B"], "R_AB": T1["R_AB"],
            "accidentals": accidentals_t1, "g2_HBT": g2_hbt_t1}]
for run in ("1", "2", "3", "ND"):
    r_a, r_b, r_ab = (entry("W1", run, q) for q in ("R_A", "R_B", "R_AB"))
    w1_rows.append({"run": run, "R_A": r_a, "R_B": r_b, "R_AB": r_ab,
                    "accidentals": r_a * r_b * window_s, "g2_HBT": g2_hbt(r_a, r_b, r_ab, window_s)})
yours_w1 = pd.DataFrame(w1_rows)
print("\nW1 - Part A, g2_HBT = R_AB / (R_A R_B Δt); rates in Hz (run ND = extra ND filter, optional)")
print(yours_w1.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
kit_runs = yours_w1.loc[yours_w1["run"].isin(["1", "2", "3"]), "g2_HBT"].dropna()
w1_mean = float(kit_runs.mean()) if len(kit_runs) else math.nan
w1_se = float(kit_runs.std(ddof=1) / math.sqrt(len(kit_runs))) if len(kit_runs) >= 2 else math.nan
print(f"runs 1-3: mean g2_HBT = {w1_mean:.4f} ± {w1_se:.4f} (standard error, {len(kit_runs)} runs); "
      f"within 3 standard errors of 1? {verdict(w1_se, lambda se: abs(w1_mean - 1) <= 3 * se)}")

# W2 - Part B: g2_PS
r_t, r_a, r_ta = (entry("W2", 1, q) for q in ("R_T", "R_A", "R_TA"))
yours_w2 = {"R_T": r_t, "R_A": r_a, "R_TA": r_ta, "accidentals": r_t * r_a * window_s,
            "g2_PS": g2_ps(r_t, r_a, r_ta, window_s)}
print("\nW2 - Part B, g2_PS = R_TA / (R_T R_A Δt); rates in Hz")
print(pd.DataFrame({"Table 2": {"R_T": rates_t2["T"], "R_A": rates_t2["A"], "R_TA": rates_t2["TA"],
                                "accidentals": accidentals_ps_t2, "g2_PS": g2_ps_t2},
                    "your run": yours_w2}).to_string(float_format=lambda v: f"{v:.3f}"))
print(f"g2_PS at least {G2_PS_MIN}? {verdict(yours_w2['g2_PS'], lambda g: g >= G2_PS_MIN)}")

# W3 - Part C: g2_GRA, sigma, the accidentals
time_s = entry("W3", 1, "time")
n = {x: entry("W3", 1, f"N_{x}") for x in ("T", "TA", "TB", "TAB")}     # the worksheet's counts
rates = {"T": ratio(n["T"], time_s), "A": entry("W3", 1, "R_A"), "B": entry("W3", 1, "R_B"),
         "TA": ratio(n["TA"], time_s), "TB": ratio(n["TB"], time_s), "TAB": ratio(n["TAB"], time_s)}
g2 = g2_gra(n["T"], n["TA"], n["TB"], n["TAB"])
sigma = gra_sigma(g2, n["TAB"])
r3, r21 = accidental_triples(rates["T"], rates["A"], rates["B"], rates["TA"], rates["TB"], window_s)
chance = (r3 + r21) * time_s
yours_w3 = {"time (s)": time_s, **{f"N_{x}": v for x, v in n.items()},
            **{f"R_{x} (Hz)": v for x, v in rates.items()},
            "g2_GRA": g2, "sigma": sigma, "(1 - g2_GRA) / sigma": ratio(1 - g2, sigma),
            "R3 (Hz)": r3, "R21 (Hz)": r21,
            "g2_acc": ratio((r3 + r21) * rates["T"], rates["TA"] * rates["TB"]),
            "chance triples": chance, "(chance - N_TAB) / sqrt(N_TAB)": ratio(chance - n["TAB"], np.sqrt(n["TAB"]))}
table_2_w3 = {"time (s)": T2["time"], **{f"N_{x}": T2[f"N_{x}"] for x in n},
              **{f"R_{x} (Hz)": v for x, v in rates_t2.items()},
              "g2_GRA": g2_gra_t2, "sigma": sigma_t2, "(1 - g2_GRA) / sigma": n_sigma_t2,
              "R3 (Hz)": r3_t2, "R21 (Hz)": r21_t2, "g2_acc": g2_acc_t2,
              "chance triples": chance_t2, "(chance - N_TAB) / sqrt(N_TAB)": distance_t2}
print("\nW3 - Part C, heralded photons (GRA) and the accidentals")
print(pd.DataFrame({"Table 2": table_2_w3, "your run": yours_w3}).to_string(float_format=lambda v: f"{v:.6g}"))
print(f"g2_GRA below 1 by more than 3 sigma? "
      f"{verdict(yours_w3['(1 - g2_GRA) / sigma'], lambda k: k > 3)};  "
      f"chance triples explain N_TAB (within 2 sigma)? "
      f"{verdict(yours_w3['(chance - N_TAB) / sqrt(N_TAB)'], lambda d: abs(d) <= 2)}")

---
## D · Does a Single Photon Interfere with Itself?

**The question first:** *the heralded photons of Part C now pass a Michelson interferometer
before detector B. Do the counts rise and fall as a mirror moves, and which wavelength do the
fringes give?*

**Why this step:** **interference**: a wave split in two and recombined adds up or cancels,
depending on the path difference. In a Michelson interferometer a beam splitter sends the
photon into two arms, mirrors send it back, and the arms recombine. The stage moves one mirror
by $x$; the light goes out and back, so the path difference is $2x$ and the phase is
$\varphi = 2\pi(2x)/\lambda = 4\pi x/\lambda$. The probability to reach the detector side is

$$P_d(x) = \frac{1 - \cos(4\pi x/\lambda)}{2} ,$$

so $R_{TB}$ rises and falls with period $\lambda/2$ in mirror position (about 405 nm for
$\lambda \approx 810$ nm), while $g^{(2)}_{\mathrm{GRA}}$ stays below 1 along the scan: the
fringes are built photon by photon. The **visibility**
$\mathcal V = (R_{\max} - R_{\min})/(R_{\max} + R_{\min})$ is 1 for perfect fringes.

Backwards, from the scan to the wavelength, with **one maximum per fringe**: count the fringe
maxima $N_{\max}$ and read the first and the last position,

$$\lambda = 2 \times (\text{mean spacing of maxima}) = \frac{2\,(x_{\text{last}} - x_{\text{first}})}{N_{\max} - 1} ,$$

then compare with 810 nm; the stage error is up to 3 % (manual §9.7). Counts scatter, so the
raw counts can peak several times inside one fringe. The cell therefore smooths the counts with
a 5-point moving average (125 nm, about a third of a period), cuts the smoothed curve at its
mean level, keeps the largest smoothed value of each fringe above the mean, and drops the
incomplete fringes at the two ends of the scan.

By default the cell analyzes the **SYNTHETIC** scan (seed 1010) with the manual's §9.7
settings: 2–18 µm in 25 nm steps, 0.3 s per point. To analyze a real scan, save it with the
same columns (`x_nm`, `counts`, `dwell_s`), put it next to this notebook and set `SCAN_FILE`.
The W4 cell after it runs the same formula on the maxima you recorded on your worksheet.

In [ ]:
"""
Part D - the Michelson scan (SYNTHETIC by default): one smoothed maximum per fringe, and the wavelength.
"""
SYNTHETIC_SCAN = f"lab10_michelson_SYNTHETIC_seed{SEED}.csv"
SCAN_FILE = SYNTHETIC_SCAN      # <- a real scan with the same columns (x_nm, counts, dwell_s)
USING_SYNTHETIC = SCAN_FILE == SYNTHETIC_SCAN
LAMBDA_NM, STAGE_ERROR = 810.0, 0.03      # expected wavelength; stage error up to 3 % (manual §9.7)
SMOOTH_POINTS, MIN_POINTS_PER_FRINGE = 5, 3

scan, notes = load_csv(SCAN_FILE)
print("\n".join(notes), "\n")
x_nm, counts, dwell_s = (to_number(scan[c]).to_numpy() for c in ("x_nm", "counts", "dwell_s"))
assert np.isfinite(x_nm).all() and np.isfinite(counts).all(), "blank or unreadable scan values"
assert np.all(np.diff(x_nm) > 0), "the scan must be in increasing x"
LABEL = f"SYNTHETIC scan, seed {SEED}" if USING_SYNTHETIC else SCAN_FILE


def smoothed_maxima(x, n, smooth_points=SMOOTH_POINTS):
    """One maximum per complete fringe; returns (maxima positions, smoothed x, smoothed counts).

    1. centred moving average of smooth_points points (only positions with a full window);
    2. each run of smoothed points above the smoothed mean is one fringe; runs touching
       either end of the smoothed curve are incomplete fringes and are dropped;
    3. the fringe maximum is the position of the largest smoothed value in its run.
    """
    half = smooth_points // 2
    smooth = np.convolve(np.asarray(n, dtype=float), np.ones(smooth_points) / smooth_points, "valid")
    x_s = np.asarray(x, dtype=float)[half: len(x) - half]
    above = smooth > smooth.mean()
    edges = np.flatnonzero(np.diff(above.astype(np.int8)))
    starts = np.concatenate(([0], edges + 1))
    stops = np.concatenate((edges + 1, [len(above)]))
    maxima = []
    for lo, hi in zip(starts, stops):
        if not above[lo] or lo == 0 or hi == len(above):
            continue                      # below the mean, or a fringe cut by an end of the scan
        if hi - lo < MIN_POINTS_PER_FRINGE:
            raise ValueError(f"a fringe of only {hi - lo} points at x = {x_s[lo]:g} nm: noise, smooth more")
        maxima.append(x_s[lo + int(np.argmax(smooth[lo:hi]))])
    return np.array(maxima), x_s, smooth


x_max, x_smooth, smooth = smoothed_maxima(x_nm, counts)
N_MAX = len(x_max)
spacing_nm = (x_max[-1] - x_max[0]) / (N_MAX - 1)
wavelength_nm = wavelength_from_maxima(x_max[0], x_max[-1], N_MAX)
deviation = (wavelength_nm - LAMBDA_NM) / LAMBDA_NM

print(f"{LABEL}: {len(x_nm)} points, x = {x_nm[0]:g}-{x_nm[-1]:g} nm")
print(f"one smoothed maximum per complete fringe: N_max = {N_MAX}")
print(f"x_first = {x_max[0]:g} nm, x_last = {x_max[-1]:g} nm, mean spacing = {spacing_nm:.2f} nm "
      f"(predicted lambda / 2 = {LAMBDA_NM / 2:g} nm)")
print(f"lambda = 2 (x_last - x_first) / (N_max - 1) = {wavelength_nm:.2f} nm:  "
      f"{100 * deviation:+.2f} % from {LAMBDA_NM:g} nm;  within {100 * STAGE_ERROR:g} %? "
      f"{verdict(deviation, lambda d: abs(d) <= STAGE_ERROR)}")
if USING_SYNTHETIC:
    assert abs(deviation) <= STAGE_ERROR

rate_khz = counts / dwell_s / 1e3
fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(x_nm / 1e3, rate_khz, ".", ms=3, color=DATA_COLOR, alpha=0.35, label="counts / dwell, per point")
ax.plot(x_smooth / 1e3, smooth / dwell_s[0] / 1e3, color=DATA_COLOR, lw=1.3,
        label=f"{SMOOTH_POINTS}-point moving average")
ax.plot(x_max / 1e3, np.interp(x_max, x_smooth, smooth) / dwell_s[0] / 1e3, "v", ms=6,
        color=PREDICTION_COLOR, label=f"one maximum per fringe (N_max = {N_MAX})")
ax.set_xlabel("mirror displacement x (µm)")
ax.set_ylabel("T&B coincidence rate (kHz)")
ax.set_title(f"{LABEL}: lambda = {wavelength_nm:.1f} nm from {N_MAX} maxima", fontsize=10)
ax.grid(axis="y", color="0.9")
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=3, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
"""
Your data, W4: the wavelength from the maxima on your worksheet (run the W1-W3 cell first).
N_max, x_first and x_last are enough; if they are blank, the recorded maxima x_max_1, ... are used.
"""
recorded = np.array([entry("W4", 1, f"x_max_{k}") for k in range(1, 21)]) * 1e9     # nm
recorded = recorded[np.isfinite(recorded)]
n_max_yours = entry("W4", 1, "N_max")
x_first_yours, x_last_yours = entry("W4", 1, "x_first") * 1e9, entry("W4", 1, "x_last") * 1e9
if np.isnan([n_max_yours, x_first_yours, x_last_yours]).any() and len(recorded) >= 2:
    print(f"N_max, x_first or x_last blank: using the {len(recorded)} recorded maxima instead "
          "(this assumes they are consecutive maxima, with none skipped)")
    n_max_yours, x_first_yours, x_last_yours = float(len(recorded)), recorded[0], recorded[-1]
if len(recorded) >= 2:
    print("spacings of the recorded maxima (nm):", np.round(np.diff(recorded), 1).tolist())

lambda_yours = wavelength_from_maxima(x_first_yours, x_last_yours, n_max_yours)
DEVIATION = f"deviation from {LAMBDA_NM:g} nm (%)"
r_max, r_min = entry("W4", 1, "R_max"), entry("W4", 1, "R_min")
yours_w4 = {"N_max": n_max_yours, "x_first (nm)": x_first_yours, "x_last (nm)": x_last_yours,
            "spacing (nm)": ratio(x_last_yours - x_first_yours, n_max_yours - 1),
            "lambda (nm)": lambda_yours, DEVIATION: 100 * ratio(lambda_yours - LAMBDA_NM, LAMBDA_NM),
            "visibility": ratio(r_max - r_min, r_max + r_min)}
scan_w4 = {"N_max": N_MAX, "x_first (nm)": x_max[0], "x_last (nm)": x_max[-1], "spacing (nm)": spacing_nm,
           "lambda (nm)": wavelength_nm, DEVIATION: 100 * deviation, "visibility": math.nan}
print("\nW4 - Part D, lambda = 2 (x_last - x_first) / (N_max - 1)")
print(pd.DataFrame({LABEL: scan_w4, "your scan": yours_w4}).to_string(float_format=lambda v: f"{v:.6g}"))
print(f"lambda within {100 * STAGE_ERROR:g} % of {LAMBDA_NM:g} nm? "
      f"{verdict(yours_w4[DEVIATION], lambda d: abs(d) <= 100 * STAGE_ERROR)}")
for where in ("at_max", "at_min", "between"):
    g2_scan = entry("W4", 1, f"g2_GRA_{where}")
    print(f"g2_GRA {where.replace('_', ' ')}: {g2_scan:.4f}, below 1? {verdict(g2_scan, lambda g: g < 1)}")

---
## Wrap-up — Back to the Bench

Every number in this notebook is one you produce at the bench:

- **Part A (W1):** an attenuated laser gives $g^{(2)}_{\mathrm{HBT}} = 1$ at every
  brightness: rare photons are not single photons;
- **Part B (W2):** $g^{(2)}_{\mathrm{PS}}$ far above 1: the crystal makes pairs;
- **Part C (W3):** heralded photons give $g^{(2)}_{\mathrm{GRA}}$ far below the classical
  bound 1, and chance coincidences in the window account for the few triples left;
- **Part D (W4):** the fringe maxima of $R_{TB}$ give $\lambda = 2\times$ spacing, while
  $g^{(2)}_{\mathrm{GRA}}$ stays below 1 along the scan.

**Replacing the SYNTHETIC scan.** Save a real scan as a CSV with the columns `x_nm` (mirror
displacement in nm), `counts` (T&B coincidences per point) and `dwell_s` (seconds per point),
upload it next to this notebook, set `SCAN_FILE` in Part D and rerun from Part D on. The format
is provisional: it will follow the export of the kit software used at i-LAMP.

**Sources.** Thorlabs, *EDU-QOP1(/M) Quantum Optics Kit User Guide*, Rev. D (2025-10-25):
§9.1 Table 1 and §9.4 Table 2 (transcribed with attribution), §9.7 (the scan settings).
Further reading: R. Hanbury Brown and R. Q. Twiss, *Nature* **177**, 27 (1956);
P. Grangier, G. Roger and A. Aspect, *Europhys. Lett.* **1**, 173 (1986).

---
## Optional (advanced) · What Would Entanglement Add?

*Nothing above needs this part. It uses numpy only, and no measurement on this kit corresponds
to it.*

**The question first:** *the course plan promises entangled photons, but this kit's crystal
makes pairs with both photons vertically polarized (declared deviation DEV-10.1). What would an
entangled pair show that the kit's pairs cannot?*

**Why this step:** **polarization** is the direction in which the light's field oscillates;
$|\theta\rangle = \cos\theta\,|H\rangle + \sin\theta\,|V\rangle$ is light polarized at the angle
$\theta$ from the horizontal. A **polarizer** at $\theta$ passes a photon in the state
$|\psi\rangle$ with probability $|\langle\theta|\psi\rangle|^2$ and absorbs it otherwise (a
direct projection: Procedure 1 of the course conventions). The kit's crystal gives
$|VV\rangle = |V\rangle\otimes|V\rangle$ pairs, "correlated in time and energy (though not
polarization-entangled)" (manual Ch. 2). Two-photon states are written **big-endian**: the
first (leftmost) factor is the photon toward T, the second the photon toward the beam splitter.

- **Entangled** = a two-photon state that cannot be written as $|\alpha\rangle\otimes|\beta\rangle$.
  $|VV\rangle$ is such a product; $|\Phi^+\rangle = (|HH\rangle + |VV\rangle)/\sqrt2$ is not.
- **Local hidden variable model** = each photon leaves the source with ready-made answers for
  every polarizer angle, and the far polarizer has no influence on them.

With a polarizer at $a$ on the photon toward T and one at $b$ on its partner, the pair passes
both with probability $P(a,b) = |(\langle a|\otimes\langle b|)\,|\psi\rangle|^2$: that is
$\sin^2 a\,\sin^2 b$ for $|VV\rangle$ and $\cos^2(a-b)/2$ for $|\Phi^+\rangle$. Four such runs,
with $a_\perp = a + 90°$, give the correlation

$$E(a,b) = P(a,b) + P(a_\perp,b_\perp) - P(a,b_\perp) - P(a_\perp,b) ,$$

and the **CHSH sum** combines four pairs of settings,

$$S = E(a,b) - E(a,b') + E(a',b) + E(a',b') .$$

Every local hidden variable model, hence every product state such as $|VV\rangle$, obeys
$|S| \le 2$ (Clauser, Horne, Shimony and Holt, *Phys. Rev. Lett.* **23**, 880 (1969)). The
cell computes the same-angle pass probability $P(\theta,\theta)$ and $S$ at the standard angles
$(a, a'; b, b') = (0°, 45°; 22.5°, 67.5°)$.

**Hardware.** Thorlabs' *EDU-QOPA1(/M) Polarization-Entanglement Extension Kit* (Rev. D,
2025-09-30) adds a crossed type-I BBO pair plus compensation crystals; it is not used in this
lab. **Anchor:** the Micius satellite sent entangled photon pairs to two ground stations
(J. Yin et al., *Science* **356**, 1140 (2017)).

In [ ]:
"""
OPTIONAL (advanced, DEV-10.1) - |VV> against |Phi+>: same-angle pass probability and CHSH (numpy only).
"""
KET_H, KET_V = np.array([1.0, 0.0]), np.array([0.0, 1.0])     # horizontal, vertical


def polarization(theta_deg):
    """|theta> = cos(theta)|H> + sin(theta)|V>."""
    theta = np.radians(theta_deg)
    return np.cos(theta) * KET_H + np.sin(theta) * KET_V


# np.kron(first, second): the first factor is the photon toward T (big-endian)
PAIRS = {"|VV>": np.kron(KET_V, KET_V),
         "|Phi+>": (np.kron(KET_H, KET_H) + np.kron(KET_V, KET_V)) / np.sqrt(2)}
assert all(abs(np.vdot(psi, psi) - 1) < 1e-12 for psi in PAIRS.values())


def joint_pass(psi, a_deg, b_deg):
    """P(a, b) = |(<a| x <b|)|psi>|^2: both photons pass their polarizers."""
    return abs(np.vdot(np.kron(polarization(a_deg), polarization(b_deg)), psi)) ** 2


def correlation(psi, a_deg, b_deg):
    """E(a, b) from four joint-pass runs, with a_perp = a + 90 degrees."""
    a_perp, b_perp = a_deg + 90, b_deg + 90
    return (joint_pass(psi, a_deg, b_deg) + joint_pass(psi, a_perp, b_perp)
            - joint_pass(psi, a_deg, b_perp) - joint_pass(psi, a_perp, b_deg))


def chsh(psi, a, a2, b, b2):
    """S = E(a, b) - E(a, b') + E(a', b) + E(a', b')."""
    return correlation(psi, a, b) - correlation(psi, a, b2) + correlation(psi, a2, b) + correlation(psi, a2, b2)


ANGLES = [0, 22.5, 45, 90]
same_angle = pd.DataFrame({name: [joint_pass(psi, t, t) for t in ANGLES] for name, psi in PAIRS.items()},
                          index=pd.Index(ANGLES, name="both polarizers at (degrees)"))
print("P(theta, theta), both photons pass:  |VV>: sin^4(theta);  |Phi+>: 1/2 at every angle")
print(same_angle.to_string(float_format=lambda v: f"{v:.5f}"))
assert np.allclose(same_angle["|VV>"], np.sin(np.radians(ANGLES)) ** 4, rtol=0, atol=1e-12)
assert np.allclose(same_angle["|Phi+>"], 0.5, rtol=0, atol=1e-12)

SETTINGS = (0, 45, 22.5, 67.5)       # (a, a'; b, b') in degrees
LHV_BOUND = 2                        # |S| <= 2 for every local hidden variable model
S = {name: chsh(psi, *SETTINGS) for name, psi in PAIRS.items()}
print(f"\nCHSH at (a, a'; b, b') = ({SETTINGS[0]}, {SETTINGS[1]}; {SETTINGS[2]}, {SETTINGS[3]}) degrees:")
for name, value in S.items():
    side = "within" if abs(value) <= LHV_BOUND + 1e-12 else "beyond"
    print(f"  {name:7s} S = {value:.6f}  ({side} the local hidden variable bound {LHV_BOUND})")
print(f"  sqrt(2) = {math.sqrt(2):.6f},  2 sqrt(2) = {2 * math.sqrt(2):.6f}")
assert abs(S["|VV>"] - math.sqrt(2)) < 1e-12 and abs(S["|Phi+>"] - 2 * math.sqrt(2)) < 1e-12

theta = np.linspace(0, 180, 361)
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(theta, [joint_pass(PAIRS["|VV>"], t, t) for t in theta], color=DATA_COLOR, lw=2,
        label="|VV>, the kit's pairs: sin^4(theta)")
ax.plot(theta, [joint_pass(PAIRS["|Phi+>"], t, t) for t in theta], color=PREDICTION_COLOR, lw=2, ls="--",
        label="|Phi+>, entangled: 1/2")
ax.set_xticks(range(0, 181, 45))
ax.set_xlabel("both polarizers at theta (degrees)")
ax.set_ylabel("P(both photons pass)")
ax.set_title("Same angle on both photons (numpy model, not a measurement)", fontsize=10)
ax.grid(axis="y", color="0.9")
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

---
## Optional · The Pair States as Qiskit Circuits

*Nothing above needs this part. The cell installs Qiskit if it is missing, and skips itself
with a one-line message if it cannot.*

**The question first:** *Qiskit prints two-qubit results as bit strings. Which bit belongs to
which photon?*

**Why this step:** a pair is two qubits: qubit 0 is the photon toward T, qubit 1 its partner,
and the course convention maps the qubit value 0 to $H$ and 1 to $V$. The lab writes bit
strings **big-endian** (qubit 0 leftmost), in the order of the state
$|\alpha\rangle\otimes|\beta\rangle$; Qiskit prints them **little-endian** (qubit 0
rightmost). `qiskit_to_lab` reverses each string: a photon toward T in $H$ with its partner in
$V$, the state $|H\rangle\otimes|V\rangle$, reads `01` in the lab's order, while Qiskit itself
prints `10`.

The cell builds three circuits: $|H\rangle\otimes|V\rangle$ (the bit-order test), the kit's
$|VV\rangle$, and $|\Phi^+\rangle$ (a Hadamard gate on qubit 0, then a CNOT from qubit 0 to
qubit 1). It checks their exact probabilities against the same states built with numpy, then
samples 1000 shots on a seeded simulator. Every measurement is in the H/V setting (polarizers
at 0° and 90°), the one setting where $|VV\rangle$ and $|\Phi^+\rangle$ give the same perfect
correlation $E = 1$: they differ only at other angles, as the numpy section showed.

In [ ]:
"""
OPTIONAL - |H>(x)|V>, |VV> and |Phi+> as 2-qubit circuits (Qiskit); bit strings through qiskit_to_lab.
Installs Qiskit if it is missing; if that fails, prints one line and skips.
"""
import importlib
import subprocess
import sys


def qiskit_available():
    try:
        importlib.import_module("qiskit")
        return True
    except ImportError:
        return False


if not qiskit_available():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qiskit"], check=False)
    importlib.invalidate_caches()

if not qiskit_available():
    print("Qiskit is not available: optional cell skipped (Parts A-D are unaffected).")
else:
    from qiskit import QuantumCircuit
    from qiskit.primitives import StatevectorSampler
    from qiskit.quantum_info import Statevector

    def qiskit_to_lab(bitstring):
        """Qiskit's bit order (qubit 0 rightmost) -> the lab's (qubit 0 leftmost): a string reversal."""
        return bitstring[::-1]

    assert qiskit_to_lab("01") == "10"
    assert all(qiskit_to_lab(qiskit_to_lab(s)) == s for s in ("00", "01", "10", "11"))

    ket_h, ket_v = np.array([1.0, 0.0]), np.array([0.0, 1.0])
    hv, vv, phi_plus = QuantumCircuit(2), QuantumCircuit(2), QuantumCircuit(2)
    hv.x(1)                      # photon toward T (qubit 0) horizontal, its partner (qubit 1) vertical
    vv.x(0)
    vv.x(1)                      # both vertical: the kit's pairs
    phi_plus.h(0)
    phi_plus.cx(0, 1)            # (|HH> + |VV>)/sqrt(2)
    CASES = {"|H>(x)|V>": (hv, np.kron(ket_h, ket_v)),
             "|VV>": (vv, np.kron(ket_v, ket_v)),
             "|Phi+>": (phi_plus, (np.kron(ket_h, ket_h) + np.kron(ket_v, ket_v)) / np.sqrt(2))}

    for name, (qc, psi) in CASES.items():
        exact = {qiskit_to_lab(k): p for k, p in Statevector(qc).probabilities_dict().items() if p > 1e-12}
        numpy_probs = {format(i, "02b"): abs(a) ** 2 for i, a in enumerate(psi) if abs(a) ** 2 > 1e-12}
        assert exact.keys() == numpy_probs.keys(), (name, exact, numpy_probs)
        assert all(abs(exact[k] - numpy_probs[k]) < 1e-12 for k in exact)

    SHOTS = 1000
    measured = []
    for qc, _ in CASES.values():
        m = qc.copy()
        m.measure_all()
        measured.append(m)
    results = StatevectorSampler(seed=SEED).run(measured, shots=SHOTS).result()
    print(f" state        Qiskit prints            lab order (qiskit_to_lab)   E(0°, 0°) from {SHOTS} shots")
    correlation_hv = {}
    for name, result in zip(CASES, results):
        raw = dict(sorted(result.data.meas.get_counts().items()))
        lab = dict(sorted((qiskit_to_lab(k), v) for k, v in raw.items()))
        same = sum(v for k, v in lab.items() if k[0] == k[1])
        correlation_hv[name] = (same - (SHOTS - same)) / SHOTS
        print(f" {name:11s}  {str(raw):23s}  {str(lab):26s}  {correlation_hv[name]:+.3f}")
        if name == "|H>(x)|V>":
            assert raw == {"10": SHOTS} and lab == {"01": SHOTS}
        elif name == "|VV>":
            assert lab == {"11": SHOTS}
        else:
            assert set(lab) <= {"00", "11"} and abs(lab.get("00", 0) / SHOTS - 0.5) < 0.07
    assert correlation_hv["|VV>"] == correlation_hv["|Phi+>"] == 1 and correlation_hv["|H>(x)|V>"] == -1
    print("bit order checked: Qiskit's 10 is the lab's 01, the state |H>(x)|V>; "
          "|VV> and |Phi+> both give E = 1 at the H/V setting.")